# Rocklin — can anything tell a designed fold from a scramble of the same residues?

**Kaggle: Accelerator = `GPU T4 x2`, Internet = `ON`.** About 80 minutes, most of it ESMFold.

---

## Why this experiment exists

The Tsuboyama run (notebooks 48 and 49) produced a clean but frustrating answer: **sequence length
alone (ρ = +0.396) beat ESMFold's confidence, ProtGPT2's activations, and every combination.**
Nothing added anything to a ruler.

Two things were wrong with that setup, and this notebook fixes both.

**Problem 1 — the wrong question.** Every Tsuboyama domain is a real, folded protein. The question
was "among things that all fold, which is *more stable*?" But the original project's finding was
about something coarser: *does this sequence fold at all?* There were no non-folding sequences in
Tsuboyama, so the thing the activations were good at had no variance to explain.

**Problem 2 — the length/composition confound ate everything.** Big proteins are more stable, so
counting residues won.

## The fix: a control that makes the confound impossible

Rocklin et al. (2017) designed ~15,000 miniproteins **and, for each one, scrambled versions of it** —
the same amino acids in a different order. Then they measured all of them.

That means for every design we have a partner sequence with:

- **identical length**
- **identical amino-acid composition**
- **different residue order**
- **its own experimental stability measurement**

Verified on the real data below: **99.9% of scrambles match their design exactly on both counts.**

So the trivial baseline that won on Tsuboyama — length plus composition — is now **mathematically
incapable of telling the two apart.** It must score exactly 50% on matched pairs, by construction.
Any model that beats chance is using **residue order**, which is to say it is using something about
protein structure.

This is the original project's question, asked against real laboratory measurements, with the
confound designed out rather than argued away.

## The headline test

For each composition-matched pair, the models see two sequences and must say which one the lab found
more stable. Chance is 50%.

| Contestant | What it sees |
|---|---|
| **A** | ESMFold pLDDT (mean, min, 10th pct) + pTM |
| **B** | ProtGPT2 activations, one layer, PCA-20 |
| **C** | A + B |
| **D** | length + amino-acid composition — **must land on 50%** |

**D is now a bug detector.** Its features are byte-identical for both members of a pair, so if D
scores anything other than 50%, the pairing is broken and nothing else in the notebook can be
trusted.

## Pre-registered readings — decided BEFORE running

| Outcome | Reading |
|---|---|
| **D ≠ 50%** | **Stop. The pairing is broken.** Debug before reading anything else. |
| High raw accuracy, balanced ≈ 50% | The model is detecting "this is a design", not grading stability. Report it as such — it is a real but different finding. |
| **C > A**, paired CI excludes zero | **The headline.** ProtGPT2 sees fold-relevant sequence order that ESMFold's confidence misses. This is the original claim, vindicated against real labels. |
| B > chance, A ≈ chance | The PLM sees foldability; ESMFold's confidence does not. Strong and surprising — scrutinise for leakage. |
| A > chance, B ≈ chance | ESMFold sees it, the PLM doesn't. A clean negative for the activation hypothesis. |
| Neither beats chance | **Neither model can distinguish a designed fold from a scramble of the same residues.** A striking, publishable negative — and a much sharper claim than the Tsuboyama result. |

Note what is *not* assumed: the design is **not** always the more stable one. In the real data the
design wins only 58.5% of the time. The label is whichever the lab actually measured as more stable,
so a model cannot win by learning "designs beat scrambles".

In [1]:
# ============================================================
#  CONFIG
# ============================================================
RANDOM_SEED  = 20260903

HF_REPO      = "AI4Protein/TAPE_Stability"   # the TAPE mirror of Rocklin et al. 2017
TOPOLOGIES   = ["EEHEE", "HEEH", "EHEE", "HHH"]   # the four designed folds
CTRL_SUFFIX  = ["_hp", "_random"]            # the composition-matched scrambles (verified below)

N_PARENTS    = 600      # ~3.1 sequences each -> ~1,860 folds -> ~56 min of ESMFold
MIN_GAP      = 0.5      # a matched pair counts only if the measured labels differ by this much
STABLE_AT    = 1.0      # Rocklin's "stable" threshold on the stability score

PLM_NAME     = "nferruz/ProtGPT2"
LAYERS       = [6, 12, 18, 24, 30]
POOL         = "mean"
N_PCA        = 20

N_FOLDS      = 5
N_REPEATS    = 5        # fewer than notebook 48: ~4x the data, and 600 groups is plenty
N_BOOT       = 2000

RUN_ESMFOLD  = True
MAX_SEQS     = None     # set to 60 for a smoke test
OUT          = "/kaggle/working"

# ============================================================
import warnings; warnings.filterwarnings("ignore")
import os, gc, json, math, time, types, collections

import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr, pearsonr
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LogisticRegressionCV
from sklearn.metrics import roc_auc_score

np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)
os.makedirs(OUT, exist_ok=True)

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def banner(t):
    print("\n" + "=" * 78); print(t); print("=" * 78)

print("CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
import transformers, sklearn, scipy
for m in (torch, transformers, sklearn, scipy, np, pd):
    print(f"  {m.__name__:14s} {m.__version__}")

CUDA: True | GPUs: 2
  torch          2.10.0+cu128
  transformers   5.0.0
  sklearn        1.6.1
  scipy          1.16.3
  numpy          2.0.2
  pandas         2.3.3


---
## Phase 1 — load, and verify the controls really are composition-matched

The whole experiment rests on one claim: a scramble has the same residues as its design, in a
different order. The cell below **checks that on every pair it builds** rather than trusting it.
If the match rate is not essentially 100%, stop.

Source: `AI4Protein/TAPE_Stability` on HuggingFace — the TAPE benchmark's mirror of Rocklin et al.
2017, three small CSVs (~7 MB total). Columns: `name`, `aa_seq`, `protein_length`, `topology`,
`parent`, `label`.

`label` is Rocklin's **stability score**: measured protease resistance minus the resistance
predicted for that sequence if it were unfolded. It is already composition-corrected by
construction, which is a second layer of protection against the confound. Above 1.0 is conventionally
"stable".

In [2]:
banner("PHASE 1 — load and verify")

from huggingface_hub import hf_hub_download

parts = []
for f in ["train.csv", "valid.csv", "test.csv"]:
    p = hf_hub_download(HF_REPO, f, repo_type="dataset")
    parts.append(pd.read_csv(p))
    print(f"  {f}: {parts[-1].shape}")
raw = pd.concat(parts, ignore_index=True)
print(f"\nTotal {len(raw):,} rows | columns: {list(raw.columns)}")
print(f"Unique parents: {raw['parent'].nunique():,}")

def suffix(nm):
    nm = str(nm)
    t = nm.split(".pdb", 1)[1] if ".pdb" in nm else "<none>"
    return t if t else "<design>"

raw["suf"] = raw["name"].map(suffix)
print("\nRow types (top 8):"); print(raw["suf"].value_counts().head(8).to_string())
print("\nStability score:"); print(raw["label"].describe().to_string())
print(f"  fraction >= {STABLE_AT} (stable): {(raw['label'] >= STABLE_AT).mean():.1%}")

# --- restrict to designs + the two verified scramble types, in the four designed folds ---
uni = raw[raw["suf"].isin(["<design>"] + CTRL_SUFFIX) & raw["topology"].isin(TOPOLOGIES)].copy()
have = uni.groupby("parent")["suf"].agg(set)
good = [p for p, s in have.items() if "<design>" in s and (set(CTRL_SUFFIX) & s)]
uni = uni[uni["parent"].isin(good)].copy()
print(f"\nParents with a design AND >=1 scramble: {len(good):,}")
print(f"Rows available: {len(uni):,}  ({len(uni)/len(good):.2f} per parent)")

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")
uni = uni[uni["aa_seq"].map(lambda s: set(str(s)) <= VALID_AA)]

# --- THE CRITICAL CHECK ---
print("\n--- verifying scrambles are composition-matched to their design ---")
n_ok_len = n_ok_comp = n_pairs = 0
for p, g in uni.groupby("parent"):
    d = g[g["suf"] == "<design>"]
    if len(d) != 1: continue
    ds = d["aa_seq"].iloc[0]
    for s in g.loc[g["suf"] != "<design>", "aa_seq"]:
        n_pairs += 1
        n_ok_len += (len(s) == len(ds))
        n_ok_comp += (collections.Counter(s) == collections.Counter(ds))
print(f"  pairs checked        : {n_pairs:,}")
print(f"  identical LENGTH     : {n_ok_len:,} ({n_ok_len/n_pairs:.2%})")
print(f"  identical COMPOSITION: {n_ok_comp:,} ({n_ok_comp/n_pairs:.2%})")
if n_ok_comp / n_pairs < 0.97:
    raise RuntimeError("Scrambles are NOT composition-matched — the experiment's premise fails. Stop.")
print("  -> premise holds: only residue ORDER differs within a pair.")


PHASE 1 — load and verify


train.csv: 0.00B [00:00, ?B/s]

  train.csv: (53614, 6)


valid.csv: 0.00B [00:00, ?B/s]

  valid.csv: (2512, 6)


test.csv: 0.00B [00:00, ?B/s]

  test.csv: (12851, 6)

Total 68,977 rows | columns: ['name', 'aa_seq', 'protein_length', 'topology', 'parent', 'label']
Unique parents: 23,040

Row types (top 8):
suf
<design>                17756
_PG_hp                  12069
_buryD                  12058
<none>                   7298
_hp                      4073
_random                  4055
_PG_hp_prottest_R14T       17
_PG_hp_prottest_K16T       16

Stability score:
count    68977.000000
mean         0.336027
std          0.630790
min         -1.970000
25%         -0.120000
50%          0.260000
75%          0.790000
max          3.400000
  fraction >= 1.0 (stable): 17.2%

Parents with a design AND >=1 scramble: 4,073
Rows available: 12,650  (3.11 per parent)

--- verifying scrambles are composition-matched to their design ---
  pairs checked        : 6,861
  identical LENGTH     : 6,850 (99.84%)
  identical COMPOSITION: 6,850 (99.84%)
  -> premise holds: only residue ORDER differs within a pair.


### Sampling down to something ESMFold can fold in one session

ESMFold runs at roughly 1.8 s per 45-residue sequence, so the full 12,650 rows would take six hours.
Sampling **600 parents** keeps every member of each sampled family together — never splitting a
matched pair — and lands around 1,860 sequences, near an hour of folding.

Sampling is stratified by topology so all four designed folds stay represented.

In [3]:
rng0 = np.random.default_rng(RANDOM_SEED)
designs = uni[uni["suf"] == "<design>"][["parent", "topology"]].drop_duplicates("parent")
per_topo = max(1, N_PARENTS // len(TOPOLOGIES))
picked = []
for t in TOPOLOGIES:
    pool = designs.loc[designs["topology"] == t, "parent"].values
    take = min(per_topo, len(pool))
    picked += list(rng0.choice(pool, size=take, replace=False))
picked = set(picked)

df = uni[uni["parent"].isin(picked)].copy().reset_index(drop=True)
if MAX_SEQS is not None:
    keep = set(list(dict.fromkeys(df["parent"]))[:max(2, MAX_SEQS // 3)])
    df = df[df["parent"].isin(keep)].reset_index(drop=True)
    print(f"*** SMOKE TEST: {df['parent'].nunique()} parents ***")

df = df.drop_duplicates("aa_seq").reset_index(drop=True)
df["length"] = df["aa_seq"].str.len()
df["is_design"] = (df["suf"] == "<design>").astype(int)

print(f"Sampled {df['parent'].nunique()} parents -> {len(df)} sequences")
print(f"Estimated ESMFold time: ~{len(df) * 1.8 / 60:.0f} min")
print("\nBy topology:"); print(df.groupby("topology")["suf"].value_counts().unstack(fill_value=0).to_string())
print("\nStability score by row type:")
print(df.groupby("suf")["label"].agg(["count", "mean", "std", "min", "max"]).to_string())
print("\nFraction stable (>= %.1f) by row type:" % STABLE_AT)
print(df.groupby("suf")["label"].apply(lambda x: (x >= STABLE_AT).mean()).to_string())
print(f"\nLength: median {df['length'].median():.0f}, range {df['length'].min()}-{df['length'].max()}")

df.to_csv(f"{OUT}/rocklin_subset.csv", index=False)
seqs = df["aa_seq"].tolist(); n = len(seqs)

Sampled 600 parents -> 1859 sequences
Estimated ESMFold time: ~56 min

By topology:
suf       <design>  _hp  _random
topology                        
EEHEE          152  149      147
EHEE           158  148      148
HEEH           156  148      148
HHH            213  148      144

Stability score by row type:
          count      mean       std   min   max
suf                                            
<design>    679  0.288954  0.586756 -1.46  1.93
_hp         593 -0.045885  0.440131 -1.53  1.47
_random     587 -0.110596  0.463353 -1.47  2.15

Fraction stable (>= 1.0) by row type:
suf
<design>    0.128130
_hp         0.011804
_random     0.017036

Length: median 43, range 43-50


---
## Phase 2 — grouping

Far simpler than Tsuboyama: the dataset ships a **`parent`** column, so families come for free — a
design and all its scrambles share one parent and always land on the same side of any split.

Two extra guards, both cheap:

1. **Identical sequences across different parents** get merged (they are the same protein).
2. A **near-duplicate screen at 90% identity.** Note this threshold is high on purpose. In Tsuboyama
   the danger was remote homology at ~30%, where identity is useless at these lengths; here the
   danger is different — accidental near-copies between independently generated designs — and 90%
   is far above any chance level, so it flags only real duplicates.

In [4]:
banner("PHASE 2 — grouping by design family")

parent_ids = {p: i for i, p in enumerate(sorted(df["parent"].unique()))}
groups = df["parent"].map(parent_ids).values.copy()

par = list(range(n))
def find(x):
    while par[x] != x: par[x] = par[par[x]]; x = par[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: par[rb] = ra

for g in pd.Series(range(n)).groupby(groups).apply(list):
    for j in g[1:]: union(g[0], j)

dupes = 0
for s, idx in pd.Series(range(n)).groupby(pd.Series(seqs)).apply(list).items():
    for j in idx[1:]:
        if find(idx[0]) != find(j): union(idx[0], j); dupes += 1
print(f"Identical sequences merged across parents: {dupes}")

try:
    from Bio.Align import PairwiseAligner, substitution_matrices
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)
    from Bio.Align import PairwiseAligner, substitution_matrices

al = PairwiseAligner(); al.mode = "global"
al.substitution_matrix = substitution_matrices.load("BLOSUM62")
al.open_gap_score = -11; al.extend_gap_score = -1

def ident(i, j):
    a = al.align(seqs[i], seqs[j])[0]
    x, y = str(a[0]), str(a[1])
    return sum(1 for u, v in zip(x, y) if u == v and u != "-") / min(len(seqs[i]), len(seqs[j]))

def kmers(s, k=4): return {s[i:i+k] for i in range(len(s) - k + 1)}
KS = [kmers(s) for s in seqs]

t0 = time.time(); extra = 0
for i in range(n):
    for j in range(i + 1, n):
        if find(i) == find(j): continue
        if len(KS[i] & KS[j]) / max(1, min(len(KS[i]), len(KS[j]))) < 0.5: continue
        if ident(i, j) >= 0.90:
            union(i, j); extra += 1
print(f"Near-duplicate links at >=90% identity: {extra}  ({time.time()-t0:.0f}s)")

df["cluster"] = [find(i) for i in range(n)]
groups = df["cluster"].values
sizes = df["cluster"].value_counts()
print(f"\n>>> {len(sizes)} family groups across {n} sequences — THIS IS THE SAMPLE SIZE <<<")
print(f"    largest: {sizes.iloc[0]} ({sizes.iloc[0]/n:.1%})   median: {sizes.median():.0f}")
if sizes.iloc[0] / n > 0.10:
    print("*** WARNING: one group holds >10% of the data. ***")
df.to_csv(f"{OUT}/rocklin_subset.csv", index=False)


PHASE 2 — grouping by design family
Identical sequences merged across parents: 0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.5 MB/s eta 0:00:00
Near-duplicate links at >=90% identity: 0  (3s)

>>> 600 family groups across 1859 sequences — THIS IS THE SAMPLE SIZE <<<
    largest: 4 (0.2%)   median: 3


---
## Phase 3 — ProtGPT2 activations

Same as before, and again deliberately **not** ESM-2: ESMFold's language trunk *is* ESM-2, so using
it here would put the same engine on both sides of the comparison.

Runs first and saves to disk, so an ESMFold crash later costs nothing.

In [5]:
banner("PHASE 3 — ProtGPT2 activations")
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(PLM_NAME)
plm = AutoModelForCausalLM.from_pretrained(PLM_NAME)
dev = "cuda" if torch.cuda.is_available() else "cpu"
plm = plm.to(dev).eval()
d_model = plm.config.n_embd
print(f"{PLM_NAME}: {plm.config.n_layer} layers, d_model={d_model}, on {dev}")

acts = {p: {L: np.zeros((n, d_model), dtype=np.float32) for L in LAYERS} for p in ("mean", "max")}
ntok = np.zeros(n, dtype=np.int32)
t0 = time.time()
for i, s in enumerate(seqs):
    ids = tok(s, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = plm(**ids, output_hidden_states=True)
    ntok[i] = ids["input_ids"].shape[1]
    for L in LAYERS:
        h = out.hidden_states[L][0]
        acts["mean"][L][i] = h.mean(0).float().cpu().numpy()
        acts["max"][L][i] = h.max(0).values.float().cpu().numpy()
    if (i + 1) % 300 == 0: print(f"  {i+1}/{n} ({time.time()-t0:.0f}s)")

print(f"Done in {time.time()-t0:.0f}s | tokens/seq median {np.median(ntok):.0f}")
np.savez_compressed(f"{OUT}/rocklin_activations.npz",
                    **{f"{p}_L{L}": acts[p][L] for p in acts for L in LAYERS},
                    n_tokens=ntok, aa_seq=np.array(seqs, dtype=object))
print(f"Saved -> {OUT}/rocklin_activations.npz")
del plm, out; clear_gpu()


PHASE 3 — ProtGPT2 activations


config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


nferruz/ProtGPT2: 36 layers, d_model=1280, on cuda
  300/1859 (16s)
  600/1859 (27s)
  900/1859 (35s)
  1200/1859 (44s)
  1500/1859 (53s)
  1800/1859 (62s)
Done in 63s | tokens/seq median 14
Saved -> /kaggle/working/rocklin_activations.npz


---
## Phase 4 — ESMFold

Same settled recipe as notebook 48: split across both T4s with `device_map="auto"` plus the
`compute_language_model_representations` patch that moves every layer's hidden state onto one device
before stacking. Requires **T4 × 2**. Checkpoints to CSV every 50 sequences.

In [6]:
banner("PHASE 4 — ESMFold")
from transformers import EsmForProteinFolding

def _patched(self, esmaa):
    device = next(self.parameters()).device
    B, L = esmaa.shape
    if self.config.esmfold_config.bypass_lm:
        return torch.zeros(B, L, self.esm_s_combine.size[0], -1, self.esm_feats, device=device)
    bosi, eosi = self.esm_dict_cls_idx, self.esm_dict_eos_idx
    bos = esmaa.new_full((B, 1), bosi)
    eos = esmaa.new_full((B, 1), self.esm_dict_padding_idx)
    esmaa = torch.cat([bos, esmaa, eos], dim=1)
    esmaa[range(B), (esmaa != 1).sum(1)] = eosi
    hs = self.esm(esmaa, attention_mask=esmaa != 1, output_hidden_states=True)["hidden_states"]
    tgt = self.esm_s_combine.device
    return torch.stack([h.to(tgt) for h in hs], dim=2)[:, 1:-1]

fold_csv = f"{OUT}/rocklin_plddt.csv"
if RUN_ESMFOLD:
    etok = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
    esm = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1",
                                               low_cpu_mem_usage=True, device_map="auto")
    esm.trunk.set_chunk_size(64); esm.eval()
    esm.compute_language_model_representations = types.MethodType(_patched, esm)
    print("ESMFold loaded.")

    rows, t0 = [], time.time()
    for i, s in enumerate(seqs):
        r = {"aa_seq": s, "mean_plddt": np.nan, "min_plddt": np.nan,
             "q10_plddt": np.nan, "ptm": np.nan, "fold_ok": 0}
        try:
            inp = etok([s], return_tensors="pt", add_special_tokens=False)
            inp = {k: v.to(esm.device) for k, v in inp.items()}
            with torch.no_grad(): o = esm(**inp)
            p = o.plddt[0].float().cpu().numpy()
            pr = p[:, 1] if p.ndim == 2 and p.shape[1] > 1 else p.reshape(len(s), -1).mean(1)
            pr = pr * (100.0 if np.nanmean(pr) <= 1.5 else 1.0)
            if not np.isnan(pr).any():
                r.update(mean_plddt=float(pr.mean()), min_plddt=float(pr.min()),
                         q10_plddt=float(np.percentile(pr, 10)), fold_ok=1)
            if getattr(o, "ptm", None) is not None:
                v = float(o.ptm.flatten()[0].item()) if torch.is_tensor(o.ptm) else float(o.ptm)
                r["ptm"] = v if not math.isnan(v) else np.nan
        except (RuntimeError, IndexError) as e:
            print(f"  [{i}] failed: {type(e).__name__}: {str(e)[:80]}"); clear_gpu()
        rows.append(r)
        if (i + 1) % 50 == 0:
            pd.DataFrame(rows).to_csv(fold_csv, index=False)
            el = time.time() - t0
            print(f"  {i+1}/{n} ok={sum(x['fold_ok'] for x in rows)} "
                  f"({el/60:.0f}m, {el/(i+1):.1f}s/seq, ~{(n-i-1)*el/(i+1)/60:.0f}m left)")
    fold_df = pd.DataFrame(rows); fold_df.to_csv(fold_csv, index=False)
    del esm; clear_gpu()
else:
    fold_df = pd.read_csv(fold_csv)

print(f"\nFolded OK: {int(fold_df['fold_ok'].sum())}/{len(fold_df)}")
print("mean_plddt:"); print(fold_df["mean_plddt"].describe().to_string())


PHASE 4 — ESMFold


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.bias   | MISSING    | 
esm.contact_head.regression.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ESMFold loaded.
  50/1859 ok=50 (2m, 1.8s/seq, ~55m left)
  100/1859 ok=100 (3m, 1.8s/seq, ~52m left)
  150/1859 ok=150 (4m, 1.8s/seq, ~51m left)
  200/1859 ok=200 (6m, 1.8s/seq, ~49m left)
  250/1859 ok=250 (7m, 1.8s/seq, ~47m left)
  300/1859 ok=300 (9m, 1.8s/seq, ~46m left)
  350/1859 ok=350 (10m, 1.8s/seq, ~44m left)
  400/1859 ok=400 (12m, 1.8s/seq, ~43m left)
  450/1859 ok=450 (13m, 1.8s/seq, ~41m left)
  500/1859 ok=500 (15m, 1.7s/seq, ~40m left)
  550/1859 ok=550 (16m, 1.7s/seq, ~38m left)
  600/1859 ok=600 (17m, 1.7s/seq, ~37m left)
  650/1859 ok=650 (19m, 1.8s/seq, ~35m left)
  700/1859 ok=700 (20m, 1.8s/seq, ~34m left)
  750/1859 ok=750 (22m, 1.8s/seq, ~32m left)
  800/1859 ok=800 (23m, 1.8s/seq, ~31m left)
  850/1859 ok=850 (25m, 1.7s/seq, ~29m left)
  900/1859 ok=900 (26m, 1.7s/seq, ~28m left)
  950/1859 ok=950 (28m, 1.7s/seq, ~26m left)
  1000/1859 ok=1000 (29m, 1.7s/seq, ~25m left)
  1050/1859 ok=1050 (30m, 1.7s/seq, ~23m left)
  1100/1859 ok=1100 (32m, 1.7s/seq, ~22m le

---
## Phase 5 — the gate

Two things before any modelling.

**Does pLDDT even separate designs from their scrambles?** If ESMFold gives a design and a shuffled
version of it the same confidence, that is already a striking result about pLDDT.

**And the sanity check that matters most:** designs should be measurably more stable than their
scrambles on average. If they are not, the dataset is not what we think it is.

In [7]:
banner("PHASE 5 — GATE")
dfa = df.merge(fold_df, on="aa_seq", how="left")
dfa = dfa[dfa["fold_ok"] == 1].reset_index(drop=True)
print(f"Analysis set: {len(dfa)} sequences, {dfa['cluster'].nunique()} family groups "
      f"({len(df) - len(dfa)} dropped — ESMFold failed)")

groups = dfa["cluster"].values
y = dfa["label"].values.astype(float)

print("\nmean pLDDT by row type:")
print(dfa.groupby("suf")["mean_plddt"].agg(["count", "mean", "std"]).to_string())
print("\nmeasured stability by row type:")
print(dfa.groupby("suf")["label"].agg(["count", "mean", "std"]).to_string())

def rep(a, b, lab):
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 10: print(f"  {lab:32s} too few"); return np.nan
    print(f"  {lab:32s} Spearman {spearmanr(a[m], b[m])[0]:+.3f}   (n={m.sum()})")
    return spearmanr(a[m], b[m])[0]

print("\nRaw correlation with measured stability score:")
rep(dfa["mean_plddt"].values, y, "mean pLDDT")
rep(dfa["ptm"].values.astype(float), y, "pTM")
rep(dfa["length"].values.astype(float), y, "sequence length")
rep(dfa["is_design"].values.astype(float), y, "is it a design (not a scramble)")

print("\n--- does ESMFold's confidence separate designs from scrambles? ---")
d_p = dfa.loc[dfa["is_design"] == 1, "mean_plddt"]
s_p = dfa.loc[dfa["is_design"] == 0, "mean_plddt"]
print(f"  designs   : mean pLDDT {d_p.mean():.1f}")
print(f"  scrambles : mean pLDDT {s_p.mean():.1f}")
print(f"  gap       : {d_p.mean() - s_p.mean():+.1f} pLDDT points")
from scipy.stats import mannwhitneyu
print(f"  Mann-Whitney p = {mannwhitneyu(d_p, s_p).pvalue:.2e}")
dfa.to_csv(f"{OUT}/rocklin_analysis_set.csv", index=False)


PHASE 5 — GATE
Analysis set: 1859 sequences, 600 family groups (0 dropped — ESMFold failed)

mean pLDDT by row type:
          count       mean        std
suf                                  
<design>    679  75.526582  10.711793
_hp         593  63.968579  11.541433
_random     587  62.381822  10.445413

measured stability by row type:
          count      mean       std
suf                                
<design>    679  0.288954  0.586756
_hp         593 -0.045885  0.440131
_random     587 -0.110596  0.463353

Raw correlation with measured stability score:
  mean pLDDT                       Spearman +0.205   (n=1859)
  pTM                              Spearman +0.246   (n=1859)
  sequence length                  Spearman +0.306   (n=1859)
  is it a design (not a scramble)  Spearman +0.313   (n=1859)

--- does ESMFold's confidence separate designs from scrambles? ---
  designs   : mean pLDDT 75.5
  scrambles : mean pLDDT 63.2
  gap       : +12.3 pLDDT points
  Mann-Whitney p = 5.3

---
## Phase 6 — the matched-pair test

The headline. For every composition-matched pair whose measured stabilities differ by at least
**0.5**, each model predicts a score for both sequences and must pick the one the lab found more
stable.

Predictions are **out-of-fold** — both members of a pair share a parent, so they are always in the
same held-out fold, and the model never trained on either.

**Watch D.** Its features are byte-identical for both members of a pair, so its predictions are
identical, every pair is a tie, and it must land on exactly 50.0%. If it doesn't, something is
broken and nothing else here means anything.

### One trap, and the fix

Among pairs with a real measured gap, **the design is the more stable one about 70% of the time**.
So a model that learned nothing about stability, but could tell a design from a scramble, would score
~70% on raw accuracy. Raw accuracy would reward the wrong skill.

The primary metric is therefore **balanced accuracy**: score the pairs where the design won and the
pairs where the *scramble* won separately, then average. A model that always bets on the design gets
100% on one half and 0% on the other &mdash; exactly 50%, no credit. Only genuine stability ranking
moves balanced accuracy above chance.

Raw accuracy is still reported, alongside the "always pick the design" oracle, so you can see how
much of any raw score is just design-detection.

In [8]:
banner("PHASE 6 — matched-pair discrimination")

AA = "ACDEFGHIKLMNPQRSTVWY"
HYDRO, AROM, POS, NEG = set("AILMFVWY"), set("FWY"), set("KR"), set("DE")
ALPHAS = np.logspace(-2, 5, 30)

def comp_feats(s):
    L = len(s); c = collections.Counter(s)
    return [c[a] / L for a in AA] + [L, sum(c[a] for a in HYDRO)/L,
            sum(c[a] for a in AROM)/L, (sum(c[a] for a in POS)-sum(c[a] for a in NEG))/L]

F_COMP = np.array([comp_feats(s) for s in dfa["aa_seq"]], dtype=np.float64)
_p = dfa["ptm"].values.astype(float)
_p = np.where(np.isfinite(_p), _p, np.nanmedian(_p[np.isfinite(_p)]) if np.isfinite(_p).any() else 0.0)
F_PLDDT = np.column_stack([dfa["mean_plddt"].values.astype(float),
                           dfa["min_plddt"].values.astype(float),
                           dfa["q10_plddt"].values.astype(float), _p])

npz = np.load(f"{OUT}/rocklin_activations.npz", allow_pickle=True)
pos = {s: i for i, s in enumerate(list(npz["aa_seq"]))}
sel = np.array([pos[s] for s in dfa["aa_seq"]])
ACT = {L: npz[f"{POOL}_L{L}"][sel].astype(np.float64) for L in LAYERS}
print(f"Feature blocks: A={F_PLDDT.shape}  B={ACT[LAYERS[0]].shape}  D={F_COMP.shape}")

KINDS = ["A", "B", "C", "D"]
NEEDS_LAYER = {"B", "C"}

def build(kind, layer, tr, te):
    btr, bte = [], []
    if kind in ("A", "C"):
        btr.append(F_PLDDT[tr]); bte.append(F_PLDDT[te])
    if kind in ("B", "C"):
        X = ACT[layer]; s0 = StandardScaler().fit(X[tr])
        k = min(N_PCA, len(tr) - 1, X.shape[1])
        p = PCA(n_components=k, svd_solver="randomized", random_state=0).fit(s0.transform(X[tr]))
        btr.append(p.transform(s0.transform(X[tr]))); bte.append(p.transform(s0.transform(X[te])))
    if kind == "D":
        btr.append(F_COMP[tr]); bte.append(F_COMP[te])
    Xtr, Xte = np.hstack(btr), np.hstack(bte)
    sc = StandardScaler().fit(Xtr)
    return sc.transform(Xtr), sc.transform(Xte)

def grouped_folds(g, k, rng):
    u, c = np.unique(g, return_counts=True)
    o = rng.permutation(len(u)); u, c = u[o], c[o]
    o = np.argsort(-c, kind="stable")
    load = np.zeros(k, dtype=int); asg = {}
    for i in o:
        f = int(np.argmin(load)); asg[u[i]] = f; load[f] += c[i]
    fo = np.array([asg[x] for x in g])
    return [(np.where(fo != f)[0], np.where(fo == f)[0]) for f in range(k)]

def pick_layer(kind, tr, rng):
    best, br = LAYERS[0], -np.inf
    ng = len(np.unique(groups[tr]))
    for L in LAYERS:
        pr = np.full(len(tr), np.nan)
        for itr, ite in grouped_folds(groups[tr], min(4, max(2, ng)), rng):
            if len(itr) < 5 or len(ite) < 2: continue
            Xa, Xb = build(kind, L, tr[itr], tr[ite])
            pr[ite] = RidgeCV(alphas=ALPHAS).fit(Xa, y[tr][itr]).predict(Xb)
        m = np.isfinite(pr)
        r = spearmanr(pr[m], y[tr][m])[0] if m.sum() > 5 else -np.inf
        if np.isfinite(r) and r > br: br, best = r, L
    return best

oof = {k: np.zeros((N_REPEATS, len(dfa))) for k in KINDS}
chosen = collections.Counter(); t0 = time.time()
for r in range(N_REPEATS):
    rng = np.random.default_rng(RANDOM_SEED + r)
    for tr, te in grouped_folds(groups, N_FOLDS, rng):
        for k in KINDS:
            L = pick_layer(k, tr, rng) if k in NEEDS_LAYER else LAYERS[0]
            if k in NEEDS_LAYER: chosen[(k, L)] += 1
            Xtr, Xte = build(k, L, tr, te)
            oof[k][r, te] = RidgeCV(alphas=ALPHAS).fit(Xtr, y[tr]).predict(Xte)
    print(f"  repeat {r+1}/{N_REPEATS} ({time.time()-t0:.0f}s)")
pred = {k: oof[k].mean(0) for k in KINDS}

LAB = {"A": "A  pLDDT", "B": "B  activations", "C": "C  pLDDT + activations",
       "D": "D  length + composition"}

print("\n--- overall out-of-fold prediction of the stability score ---")
for k in KINDS:
    print(f"  {LAB[k]:26s} Spearman {spearmanr(pred[k], y)[0]:+.3f}")
print("\nLayers chosen by the nested inner CV:")
for (k, L), c in sorted(chosen.items()):
    print(f"  {k}: layer {L:>2d} chosen {c:>3d}/{N_REPEATS*N_FOLDS}")


PHASE 6 — matched-pair discrimination
Feature blocks: A=(1859, 4)  B=(1859, 1280)  D=(1859, 24)
  repeat 1/5 (79s)
  repeat 2/5 (160s)
  repeat 3/5 (244s)
  repeat 4/5 (321s)
  repeat 5/5 (398s)

--- overall out-of-fold prediction of the stability score ---
  A  pLDDT                   Spearman +0.258
  B  activations             Spearman +0.314
  C  pLDDT + activations     Spearman +0.308
  D  length + composition    Spearman +0.168

Layers chosen by the nested inner CV:
  B: layer 18 chosen  15/25
  B: layer 24 chosen   7/25
  B: layer 30 chosen   3/25
  C: layer 18 chosen  16/25
  C: layer 24 chosen   5/25
  C: layer 30 chosen   4/25


In [9]:
banner("PHASE 6b — the headline: composition-matched pairs")

idx_of = {s: i for i, s in enumerate(dfa["aa_seq"])}
pairs = []
for p, g in dfa.groupby("parent"):
    d = g[g["is_design"] == 1]
    if len(d) != 1: continue
    di = idx_of[d["aa_seq"].iloc[0]]
    for _, r in g[g["is_design"] == 0].iterrows():
        si = idx_of[r["aa_seq"]]
        if abs(y[di] - y[si]) >= MIN_GAP:
            pairs.append((di, si, p))
print(f"Composition-matched pairs with a measured gap >= {MIN_GAP}: {len(pairs)}")
print(f"  (from {dfa['parent'].nunique()} parents)")
if len(pairs) < 50:
    print("*** WARNING: very few pairs — lower MIN_GAP or raise N_PARENTS. ***")

pair_parent = np.array([p for _, _, p in pairs])
truth = np.array([1 if y[a] > y[b] else 0 for a, b, _ in pairs])
print(f"  the design is the more stable one in {truth.mean():.1%} of pairs")

def _hits(pr, P, T):
    out = []
    for (a, b, _), t in zip(P, T):
        d = pr[a] - pr[b]
        out.append(0.5 if d == 0 else float((d > 0) == bool(t)))
    return np.array(out)

ALL = np.arange(len(pairs))

def scores(pr, sub=None):
    # Returns (raw accuracy, balanced accuracy). Balanced averages the design-won and
    # scramble-won halves, so "always bet on the design" scores exactly 50%.
    idx = ALL if sub is None else sub
    P = [pairs[i] for i in idx]; T = truth[idx]
    h = _hits(pr, P, T)
    raw = float(h.mean())
    dw, sw = h[T == 1], h[T == 0]
    if len(dw) == 0 or len(sw) == 0:
        return raw, float("nan")
    return raw, float((dw.mean() + sw.mean()) / 2)

oracle = float(truth.mean())   # "always pick the design" raw accuracy
print(f"\nReference: always betting on the design scores {oracle:.1%} raw, 50.0% balanced.")
print(f"  design-won pairs: {int(truth.sum())}   scramble-won pairs: {int((1-truth).sum())}")

uniq_par = np.unique(pair_parent)
by_par = {p: np.where(pair_parent == p)[0] for p in uniq_par}
rngb = np.random.default_rng(RANDOM_SEED + 7)
boot_raw = {k: [] for k in KINDS}
boot_bal = {k: [] for k in KINDS}
for _ in range(N_BOOT):
    pick = rngb.choice(uniq_par, size=len(uniq_par), replace=True)
    sub = np.concatenate([by_par[p] for p in pick])
    for k in KINDS:
        r, b = scores(pred[k], sub)
        boot_raw[k].append(r)
        if np.isfinite(b): boot_bal[k].append(b)

print(f"\n{'contestant':28s} {'balanced':>9s} {'95% CI':>18s} {'raw':>7s}   verdict")
acc, bal, ci, ci_raw = {}, {}, {}, {}
for k in KINDS:
    acc[k], bal[k] = scores(pred[k])
    vb = np.array(boot_bal[k]); vr = np.array(boot_raw[k])
    ci[k] = tuple(np.percentile(vb, [2.5, 97.5]))
    ci_raw[k] = tuple(np.percentile(vr, [2.5, 97.5]))
    beats = ci[k][0] > 0.5
    print(f"  {LAB[k]:26s} {bal[k]:>8.1%}  [{ci[k][0]:.1%}, {ci[k][1]:.1%}] {acc[k]:>7.1%}   "
          f"{'BEATS CHANCE' if beats else 'chance'}")

print("\nAccuracy split by which side actually won (a design-detector is lopsided):")
for k in KINDS:
    P = [pairs[i] for i in ALL]
    h = _hits(pred[k], P, truth)
    print(f"  {LAB[k]:26s} design-won {h[truth==1].mean():.1%}   scramble-won {h[truth==0].mean():.1%}")

print("\nPaired differences in BALANCED accuracy (same bootstrap resample):")
diffs = {}
for name, (x, z) in {"C - A": ("C", "A"), "B - A": ("B", "A"), "B - D": ("B", "D")}.items():
    v = np.array(boot_bal[x]) - np.array(boot_bal[z])
    lo, hi = np.percentile(v, [2.5, 97.5])
    diffs[name] = (float(v.mean()), float(lo), float(hi), float((v > 0).mean()))
    print(f"  {name:10s} {v.mean():+.3f}  [{lo:+.3f}, {hi:+.3f}]  P(>0)={float((v>0).mean()):.3f}"
          + ("  *" if lo > 0 else ""))


PHASE 6b — the headline: composition-matched pairs
Composition-matched pairs with a measured gap >= 0.5: 433
  (from 600 parents)
  the design is the more stable one in 65.6% of pairs

Reference: always betting on the design scores 65.6% raw, 50.0% balanced.
  design-won pairs: 284   scramble-won pairs: 149

contestant                    balanced             95% CI     raw   verdict
  A  pLDDT                      48.0%  [44.8%, 51.4%]   60.3%   chance
  B  activations                51.4%  [48.7%, 54.1%]   64.7%   chance
  C  pLDDT + activations        49.9%  [47.3%, 52.5%]   63.5%   chance
  D  length + composition       50.0%  [50.0%, 50.0%]   50.0%   chance

Accuracy split by which side actually won (a design-detector is lopsided):
  A  pLDDT                   design-won 87.3%   scramble-won 8.7%
  B  activations             design-won 94.0%   scramble-won 8.7%
  C  pLDDT + activations     design-won 93.7%   scramble-won 6.0%
  D  length + composition    design-won 50.0%   scrambl

---
## Phase 6c — is the residual real, or is it noise?

Balanced accuracy leans on the pairs where a **scramble beat its own design**. There are two very
different reasons a model might fail on those:

1. **The models can't see it.** The reversals are real, and neither pLDDT nor the activations resolve
   them. That is a finding about the models.
2. **There is nothing to see.** Some reversals are measurement noise, and no predictor — however
   good — could ever call them. That would cap balanced accuracy near 50% for *everyone*, and the
   honest claim would have to be narrower.

These are separable. Raise the measured gap a pair must clear. **If the reversals are real, a wider
gap makes them easier to call and balanced accuracy should climb.** If they are noise, it stays
pinned at 50% no matter how wide the gap gets.

This is the check that decides how strongly the headline can be stated, so it runs in the same
session rather than waiting on a re-run.

In [10]:
banner("PHASE 6c — gap sweep: real reversals, or noise?")

def build_pairs(gap):
    P, T = [], []
    for p_, g in dfa.groupby("parent"):
        d = g[g["is_design"] == 1]
        if len(d) != 1: continue
        di = idx_of[d["aa_seq"].iloc[0]]
        for _, r in g[g["is_design"] == 0].iterrows():
            si = idx_of[r["aa_seq"]]
            if abs(y[di] - y[si]) >= gap:
                P.append((di, si, p_)); T.append(1 if y[di] > y[si] else 0)
    return P, np.array(T)

def bal_of(pr, P, T, sub=None):
    if sub is not None:
        P = [P[i] for i in sub]; T = T[sub]
    h = np.array([0.5 if pr[a] - pr[b] == 0 else float((pr[a] - pr[b] > 0) == bool(t))
                  for (a, b, _), t in zip(P, T)])
    dw, sw = h[T == 1], h[T == 0]
    if len(dw) == 0 or len(sw) == 0: return float("nan")
    return float((dw.mean() + sw.mean()) / 2)

GAPS = [0.25, 0.5, 0.75, 1.0, 1.25]
N_BOOT_SWEEP = 1000
MIN_PAIRS = 40

print("Reading: if the reversals are REAL, balanced accuracy should CLIMB with the gap.")
print("If they are measurement noise, it stays pinned near 50%.\n")
print(f"{'gap':>5s} {'pairs':>6s} {'scr-won':>8s} {'oracle':>7s}   "
      + "  ".join(f"{k:>17s}" for k in KINDS))

sweep = {}
rs = np.random.default_rng(RANDOM_SEED + 21)
for gp in GAPS:
    P, T = build_pairs(gp)
    if len(P) < MIN_PAIRS or T.sum() < 5 or (1 - T).sum() < 5:
        print(f"{gp:>5.2f} {len(P):>6d}   too few usable pairs — stopping the sweep here")
        break
    par = np.array([x[2] for x in P])
    up = np.unique(par); byp = {q: np.where(par == q)[0] for q in up}
    cells, rec = [], {}
    for k in KINDS:
        pt = bal_of(pred[k], P, T)
        bs = []
        for _ in range(N_BOOT_SWEEP):
            pick = rs.choice(up, size=len(up), replace=True)
            sub = np.concatenate([byp[q] for q in pick])
            v = bal_of(pred[k], P, T, sub)
            if np.isfinite(v): bs.append(v)
        lo, hi = np.percentile(bs, [2.5, 97.5])
        rec[k] = dict(balanced=pt, ci=[float(lo), float(hi)])
        cells.append(f"{pt:>6.1%} [{lo:>4.0%},{hi:>4.0%}]")
    sweep[gp] = dict(n_pairs=len(P), n_scramble_won=int((1 - T).sum()),
                     oracle=float(T.mean()), by_arm=rec)
    print(f"{gp:>5.2f} {len(P):>6d} {int((1-T).sum()):>8d} {T.mean():>6.1%}   "
          + "  ".join(cells))

# ---- read the trend ----
print()
gs = sorted(sweep)
if len(gs) < 2:
    print("Not enough gap levels to read a trend.")
    sweep_verdict = "inconclusive"
else:
    best_first = max(sweep[gs[0]]["by_arm"][k]["balanced"] for k in ("A", "B", "C"))
    best_last  = max(sweep[gs[-1]]["by_arm"][k]["balanced"] for k in ("A", "B", "C"))
    clearing = [(g, k) for g in gs for k in ("A", "B", "C")
                if sweep[g]["by_arm"][k]["ci"][0] > 0.5]
    rises = best_last > best_first + 0.05
    print(f"Best of A/B/C at gap {gs[0]}: {best_first:.1%}   at gap {gs[-1]}: {best_last:.1%}"
          f"   (change {best_last - best_first:+.1%})")
    if clearing:
        print("Arms clearing chance (CI above 50%): "
              + ", ".join(f"{k} at gap {g}" for g, k in clearing))

    # LEVEL is read before TREND: an arm that clears chance carries information whether or
    # not widening the gap changes anything.
    if clearing and rises:
        sweep_verdict = "real_and_partly_visible"
        print("\n-> The reversals are REAL and partly visible: some arm clears chance, and")
        print("   widening the gap lifts accuracy further. The models resolve large stability")
        print("   differences but not small ones. State the claim at that resolution.")
    elif clearing:
        sweep_verdict = "signal_at_all_gaps"
        print("\n-> SIGNAL PRESENT. At least one arm beats chance on composition-matched pairs,")
        print("   and it does so at every gap rather than only on the easy ones. That is a real")
        print("   order-dependent stability signal, not an artifact of effect size.")
    elif rises:
        sweep_verdict = "real_trend_underpowered"
        print("\n-> Accuracy climbs with the gap but no level clears chance on its own. Suggestive")
        print("   that the reversals are real and the models see a little; underpowered to prove it.")
        print("   Raise N_PARENTS and re-run before drawing a conclusion.")
    else:
        sweep_verdict = "flat"
        print("\n-> FLAT. Nothing clears chance at any gap, and widening the gap does not help,")
        print("   so this is not a resolution problem: the models carry no information about which")
        print("   member of a matched pair is more stable, at any effect size this dataset offers.")
        print("   Note in the write-up that a noise floor on the reversals cannot be fully excluded")
        print("   from these data alone.")


PHASE 6c — gap sweep: real reversals, or noise?
Reading: if the reversals are REAL, balanced accuracy should CLIMB with the gap.
If they are measurement noise, it stays pinned near 50%.

  gap  pairs  scr-won  oracle                   A                  B                  C                  D
 0.25    724      258  64.4%    50.1% [ 48%, 53%]   51.5% [ 50%, 54%]   50.6% [ 49%, 53%]   50.0% [ 50%, 50%]
 0.50    433      149  65.6%    48.0% [ 45%, 52%]   51.4% [ 49%, 54%]   49.9% [ 47%, 52%]   50.0% [ 50%, 50%]
 0.75    253       75  70.4%    47.2% [ 43%, 51%]   52.4% [ 49%, 56%]   49.9% [ 47%, 53%]   50.0% [ 50%, 50%]
 1.00    134       37  72.4%    46.0% [ 41%, 52%]   53.9% [ 49%, 60%]   51.2% [ 48%, 55%]   50.0% [ 50%, 50%]
 1.25     71       16  77.5%    49.9% [ 41%, 60%]   58.5% [ 50%, 69%]   54.4% [ 47%, 64%]   50.0% [ 50%, 50%]

Best of A/B/C at gap 0.25: 51.5%   at gap 1.25: 58.5%   (change +7.0%)

-> Accuracy climbs with the gap but no level clears chance on its own. Suggestive


---
## Phase 7 — verdict

In [11]:
banner("PHASE 7 — verdict against the pre-registered table")

print(f"n = {len(dfa)} sequences, {dfa['cluster'].nunique()} family groups, {len(pairs)} matched pairs")
print(f"Label: Rocklin stability score  |  pairs matched on length AND composition\n")
print(f"{'contestant':28s} {'balanced':>9s} {'95% CI':>18s} {'raw':>7s}")
for k in KINDS:
    print(f"  {LAB[k]:26s} {bal[k]:>8.1%}  [{ci[k][0]:.1%}, {ci[k][1]:.1%}] {acc[k]:>7.1%}")
print(f"\n  'always pick the design' oracle: {oracle:.1%} raw, 50.0% balanced")

d_off = abs(bal["D"] - 0.5)
print("\n" + "-" * 78)
if d_off > 0.02:
    print(f"STOP — BROKEN PAIRING. D scored {bal['D']:.1%} balanced, not 50%.")
    print("D's features are identical for both members of a composition-matched pair, so it")
    print("cannot do better than chance unless the pairs are not actually matched. Debug the")
    print("pairing before reading anything else on this page.")
else:
    print(f"Sanity check passed: D landed on {bal['D']:.1%} balanced, as it must.\n")
    for k in ("A", "B", "C"):
        if ci[k][0] <= 0.5 < ci_raw[k][0]:
            print(f"NOTE: {LAB[k]} beats chance on RAW accuracy but not balanced — it is")
            print("      detecting 'this is a design', not grading stability. Report it that way.\n")
    beats = {k: ci[k][0] > 0.5 for k in ("A", "B", "C")}
    ca = diffs["C - A"]
    if beats["C"] and ca[1] > 0:
        print("HEADLINE RESULT. ProtGPT2's activations add real, order-dependent information about")
        print("foldability on top of ESMFold's confidence — measured against a laboratory outcome,")
        print("with length and composition held identical by construction. This is the original")
        print("project's claim, vindicated without the circularity and without the confound.")
    elif beats["B"] and not beats["A"]:
        print("STRONG AND SURPRISING. The PLM separates designed folds from their scrambles and")
        print("ESMFold's confidence does not. Scrutinise for leakage before believing it: check the")
        print("group count and the near-duplicate screen.")
    elif beats["A"] and not beats["B"]:
        print("CLEAN NEGATIVE FOR THE PLM. ESMFold's confidence tracks real foldability; ProtGPT2's")
        print("activations do not. The activations' earlier apparent skill was an artifact of being")
        print("trained against pLDDT itself.")
    elif beats["A"] and beats["B"]:
        print("BOTH WORK, neither clearly better. Report the paired C - A difference and its CI as")
        print("the result; this bounds what the activations add over an existing structure score.")
    else:
        print("STRIKING NEGATIVE. Neither ESMFold's confidence nor ProtGPT2's activations can tell a")
        print("designed fold from a shuffle of the same amino acids, judged against real")
        print("measurements. Given how central pLDDT is as a proxy for 'will this fold', that is a")
        print("substantive claim in its own right, and a stronger paper than a marginal positive.")

print(f"\nGap sweep: {sweep_verdict}")
if sweep_verdict == "flat":
    print("  Widening the required gap did not help, so the failure is not about resolution.")
elif sweep_verdict.startswith("real"):
    print("  Accuracy improves on larger measured differences — the models see coarse")
    print("  stability differences even though they miss fine ones. State the claim at that")
    print("  resolution rather than as a flat negative.")
print("-" * 78)

res = dict(n=len(dfa), groups=int(dfa["cluster"].nunique()), n_pairs=len(pairs),
           min_gap=MIN_GAP, n_parents=int(dfa["parent"].nunique()),
           design_more_stable_frac=float(truth.mean()),
           pair_accuracy_raw={k: float(acc[k]) for k in KINDS},
           pair_accuracy_balanced={k: float(bal[k]) for k in KINDS},
           pair_ci_balanced={k: [float(x) for x in ci[k]] for k in KINDS},
           pair_ci_raw={k: [float(x) for x in ci_raw[k]] for k in KINDS},
           design_oracle_raw=oracle,
           gap_sweep={str(g): v for g, v in sweep.items()},
           gap_sweep_verdict=sweep_verdict,
           paired_diffs=diffs,
           overall_spearman={k: float(spearmanr(pred[k], y)[0]) for k in KINDS},
           chosen_layers={f"{k}_L{L}": c for (k, L), c in chosen.items()},
           versions={m.__name__: m.__version__ for m in (torch, transformers, sklearn, scipy, np, pd)})
with open(f"{OUT}/rocklin_results.json", "w") as f: json.dump(res, f, indent=2)

outp = dfa[["aa_seq", "name", "parent", "topology", "suf", "is_design", "length", "cluster",
            "label", "mean_plddt", "min_plddt", "q10_plddt", "ptm"]].copy()
for k in KINDS: outp[f"pred_{k}"] = pred[k]
outp.to_csv(f"{OUT}/rocklin_predictions.csv", index=False)

print(f"\nSaved:\n  {OUT}/rocklin_results.json\n  {OUT}/rocklin_predictions.csv"
      f"\n  {OUT}/rocklin_analysis_set.csv\n  {OUT}/rocklin_plddt.csv"
      f"\n  {OUT}/rocklin_activations.npz")
print("\n*** Hit SAVE VERSION so these persist — an interactive run discards /kaggle/working. ***")


PHASE 7 — verdict against the pre-registered table
n = 1859 sequences, 600 family groups, 433 matched pairs
Label: Rocklin stability score  |  pairs matched on length AND composition

contestant                    balanced             95% CI     raw
  A  pLDDT                      48.0%  [44.8%, 51.4%]   60.3%
  B  activations                51.4%  [48.7%, 54.1%]   64.7%
  C  pLDDT + activations        49.9%  [47.3%, 52.5%]   63.5%
  D  length + composition       50.0%  [50.0%, 50.0%]   50.0%

  'always pick the design' oracle: 65.6% raw, 50.0% balanced

------------------------------------------------------------------------------
Sanity check passed: D landed on 50.0% balanced, as it must.

NOTE: A  pLDDT beats chance on RAW accuracy but not balanced — it is
      detecting 'this is a design', not grading stability. Report it that way.

NOTE: B  activations beats chance on RAW accuracy but not balanced — it is
      detecting 'this is a design', not grading stability. Report it that